In [ ]:
!pip install unsloth-zoo
!pip install --no-deps "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl==0.8.6" peft accelerate bitsandbytes
!pip install -U pyarrow datasets # Added this to force the correct versions

import os
os._exit(00)

# 2. Re-login to Hugging Face (Kaggle also has a 'Secrets' tool like Colab)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HuggingFace Token")
login(hf_token)

  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-89skeheh/unsloth_91e05323083d4384a1f5108e49f437e4
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-89skeheh/unsloth_91e05323083d4384a1f5108e49f437e4
  Resolved https://github.com/unslothai/unsloth.git to commit 01f2e289a7376a3a4d71a2e39bed025a72df0273
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully 

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Move this to the top!
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from unsloth import FastLanguageModel

import torch
import gc
import pandas as pd
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

# 2. Configuration & Paths (Check your dataset name here!)
train_path = "/kaggle/input/datasets/alexwenxuan/medical-qa/train_split.jsonl"
eval_path = "/kaggle/input/datasets/alexwenxuan/medical-qa/eval_split.jsonl"

max_seq_length = 512 
dtype = None 
load_in_4bit = True 

# 3. Load Model & Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 4. Data Preparation
prompt_template = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}<|end_of_text|>"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for inst, inp, out in zip(instructions, inputs, outputs):
        text = prompt_template.format(instruction=inst, input=inp, output=out)
        texts.append(text)
    return { "text" : texts, }

train_dataset = load_dataset("json", data_files=train_path, split="train")
eval_dataset = load_dataset("json", data_files=eval_path, split="train")

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
eval_dataset = eval_dataset.map(formatting_prompts_func, batched = True)

# 5. The Remaining Grid Search (Configs 8, 9, 10)
# We focus on the high-rank configurations you didn't finish
configs = [
    {"rank": 8, "alpha": 16, "lr": 2e-4},
    {"rank": 8, "alpha": 16, "lr": 1e-4},
    {"rank": 16, "alpha": 32, "lr": 2e-4},
    {"rank": 16, "alpha": 32, "lr": 1e-4},
    {"rank": 16, "alpha": 32, "lr": 5e-5},
    {"rank": 32, "alpha": 64, "lr": 2e-4},
    {"rank": 32, "alpha": 64, "lr": 1e-4},
    {"rank": 32, "alpha": 64, "lr": 5e-5},
    {"rank": 64, "alpha": 128, "lr": 1e-4},
    {"rank": 64, "alpha": 128, "lr": 5e-5},
]

results = []

for i, config in enumerate(configs):
    print(f"\nProcessing Config {i+8}...") # Labeling for your report
    
    peft_model = FastLanguageModel.get_peft_model(
        model,
        r = config["rank"],
        lora_alpha = config["alpha"],
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                          "gate_proj", "up_proj", "down_proj",],
        lora_dropout = 0,
        bias = "none",
        use_gradient_checkpointing = "unsloth",
        random_state = 42,
    )

    trainer = SFTTrainer(
        model = peft_model,
        tokenizer = tokenizer,
        train_dataset = train_dataset,
        eval_dataset = eval_dataset,
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        args = TrainingArguments(
            per_device_train_batch_size = 2,
            gradient_accumulation_steps = 4,
            max_steps = 60,
            learning_rate = config["lr"],
            fp16 = not torch.cuda.is_bf16_supported(),
            bf16 = torch.cuda.is_bf16_supported(),
            logging_steps = 10,
            eval_strategy = "steps",
            eval_steps = 20,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            output_dir = f"outputs/config_{i+8}",
            report_to = "none", # Skips the wandb prompt!
        ),
    )

    trainer.train()
    eval_metrics = trainer.evaluate()
    
    train_loss = next((log['loss'] for log in reversed(trainer.state.log_history) if 'loss' in log), None)
    
    results.append({
        "Config": i+8,
        "Rank": config["rank"],
        "Alpha": config["alpha"],
        "Learning_Rate": config["lr"],
        "Train_Loss": train_loss,
        "Eval_Loss": eval_metrics['eval_loss']
    })
    
    del trainer, peft_model
    gc.collect()
    torch.cuda.empty_cache()

# 6. Save results locally in Kaggle
df = pd.DataFrame(results)
df.to_csv("kaggle_grid_results.csv", index=False)
print("✅ Done! Final results saved to kaggle_grid_results.csv")

In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if "config_10" in dirname:
            print(os.path.join(dirname, filename))

/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/adapter_model.safetensors
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/trainer_state.json
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/training_args.bin
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/adapter_config.json
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/README.md
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/tokenizer.json
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/tokenizer_config.json
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/scaler.pt
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60/scheduler.pt
/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/c

In [6]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
import pandas as pd

# 1. Configuration & Path Fix
max_seq_length = 512
# We point directly to the checkpoint folder revealed by your search
adapter_path = "/kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60" 

# 2. Load Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. Apply Fine-Tuned Adapter
model = FastLanguageModel.for_inference(model) # Enable 2x faster inference
model.load_adapter(adapter_path)
tokenizer.padding_side = "right" # Prevents generation issues with Llama-3

# 4. Prepare Evaluation Data
eval_path = "/kaggle/input/datasets/alexwenxuan/medical-qa/eval_split.jsonl" 
eval_dataset = load_dataset("json", data_files=eval_path, split="train")

# Select 30 samples (Seed 42 ensures reproducibility for your report)
samples = eval_dataset.shuffle(seed=42).select(range(30)) 

results = []
print(f"🚀 Starting inference on 30 clinical samples using: {adapter_path}")

# 5. Inference Loop
for i, example in enumerate(samples):
    # Professional prompt formatting to match your training style
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    # Generate response with strictly controlled temperature for clinical accuracy
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 150, 
        use_cache = True,
        temperature = 0.1, # Keep it deterministic for medical safety
        top_p = 0.9
    )
    
    # Clean the output string
    full_response = tokenizer.batch_decode(outputs)[0]
    # Extract only the newly generated text after the prompt
    generated_text = full_response.split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip()
    
    results.append({
        "ID": i + 1,
        "Instruction": example['instruction'],
        "Clinical_Input": example['input'],
        "Doctor_Ground_Truth": example['output'],
        "AI_FineTuned_Response": generated_text
    })
    
    if (i + 1) % 5 == 0:
        print(f"✅ Processed {i+1}/30 samples...")

# 6. Save Final Report Data
df_eval = pd.DataFrame(results)
df_eval.to_csv("final_clinical_evaluation.csv", index=False)

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating train split: 0 examples [00:00, ? examples/s]

🚀 Starting inference on 30 clinical samples using: /kaggle/input/datasets/alexwenxuan/medical-qa-dataset/outputs/config_10/checkpoint-60
✅ Processed 5/30 samples...
✅ Processed 10/30 samples...
✅ Processed 15/30 samples...
✅ Processed 20/30 samples...
✅ Processed 25/30 samples...
✅ Processed 30/30 samples...

MISSION ACCOMPLISHED
1. Download 'final_clinical_evaluation.csv' from the Output sidebar.
2. Use this data for Slide 4: Qualitative Analysis.


In [ ]:
!pip install datasets==4.3.0
import os
os._exit(00)

  Using cached datasets-4.3.0-py3-none-any.whl.metadata (18 kB)
Using cached datasets-4.3.0-py3-none-any.whl (506 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.5.0
    Uninstalling datasets-4.5.0:
      Successfully uninstalled datasets-4.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.2.1 requires trl!=0.19.0,<=0.24.0,>=0.18.2, but you have trl 0.8.6 which is incompatible.


In [2]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
import pandas as pd

# 1. Load the RAW Base Model (No adapter!)
max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# Enable native 2x faster inference and fix padding
FastLanguageModel.for_inference(model) 
tokenizer.padding_side = "right"

# 2. Load the EXACT same 30 evaluation questions
eval_path = "/kaggle/input/datasets/alexwenxuan/medical-qa/eval_split.jsonl"
eval_dataset = load_dataset("json", data_files=eval_path, split="train")
samples = eval_dataset.shuffle(seed=42).select(range(30)) 

results = []
print("🚀 Starting inference on Base Llama-3 (Untrained)...")

# 3. Inference Loop
for i, example in enumerate(samples):
    prompt = f"### Instruction:\n{example['instruction']}\n\n### Input:\n{example['input']}\n\n### Response:\n"
    
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
    # Generate response
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 150, 
        use_cache = True,
        temperature = 0.1, 
        top_p = 0.9
    )
    
    full_response = tokenizer.batch_decode(outputs)[0]
    generated_text = full_response.split("### Response:\n")[-1].replace("<|end_of_text|>", "").strip()
    
    results.append({
        "ID": i + 1,
        "Clinical_Input": example['input'],
        "Base_Model_Response": generated_text
    })
    
    if (i + 1) % 5 == 0:
        print(f"✅ Processed {i+1}/30 samples...")

# 4. Export to CSV
df_base = pd.DataFrame(results)
df_base.to_csv("base_model_responses.csv", index=False)
print("\n✅ Done! Download 'base_model_responses.csv'")

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Generating train split: 0 examples [00:00, ? examples/s]

🚀 Starting inference on Base Llama-3 (Untrained)...
✅ Processed 5/30 samples...
✅ Processed 10/30 samples...
✅ Processed 15/30 samples...
✅ Processed 20/30 samples...
✅ Processed 25/30 samples...
✅ Processed 30/30 samples...

✅ Done! Download 'base_model_responses.csv'
